In [8]:
#loading required python classes and packages
from confluent_kafka import Consumer, KafkaError, KafkaException, Producer, TopicPartition
from confluent_kafka import Producer
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

In [21]:
#loading & displaying dataset values
dataset = pd.read_csv("Dataset/BankingTransactionDataSet.csv")
dataset

,transaction_id,sender_account_id,recipient_account_id,amount,payment_mode,timestamp,sender_location,recipient_location,device_fingerprint,transaction_frequency,is_fraud
0,T12345,SBIN00001,SBIN00010,4500.75,UPI,01-01-2025 08:15,Maharashtra,Karnataka,DF001,5,0
1,T12346,HDFC00002,ICICI00011,250000.00,RTGS,01-01-2025 14:30,Delhi,Tamil Nadu,DF002,1,1
2,T12347,ICICI00003,AXIS00012,75.00,UPI,02-01-2025 09:45,Gujarat,Maharashtra,DF003,12,0
3,T12348,AXIS00004,HDFC00013,5000.00,NEFT,02-01-2025 18:20,Kerala,Uttar Pradesh,DF004,3,1
4,T12349,PNB00005,PNB00014,300.00,IMPS,03-01-2025 07:10,Rajasthan,Rajasthan,DF005,7,0


In [22]:
#dataset cleaning and processing by converting non-numeric values to numeric values
label_encoder = []
columns = dataset.columns
types = dataset.dtypes.values
for j in range(len(types)):
    name = types[j]
    if name == 'object': #finding column with object type
        le = LabelEncoder()
        dataset[columns[j]] = pd.Series(le.fit_transform(dataset[columns[j]].astype(str)))#encode all str columns to numeric
        label_encoder.append([columns[j], le])
dataset.fillna(dataset.mean(), inplace = True)#replace missing values with meaan if exists
dataset

,transaction_id,sender_account_id,recipient_account_id,amount,payment_mode,timestamp,sender_location,recipient_location,device_fingerprint,transaction_frequency,is_fraud
0,0,4,4,4500.75,3,0,3,0,0,5,0
1,1,1,2,250000.00,2,1,0,3,1,1,1
2,2,2,0,75.00,3,2,1,1,2,12,0
3,3,0,1,5000.00,1,3,2,4,3,3,1
4,4,3,3,300.00,0,4,4,2,4,7,0


In [23]:
#extracting training features and target label
Y = dataset['is_fraud'].ravel()
dataset.drop(['is_fraud'], axis = 1,inplace=True)
X = dataset.values
scaler = StandardScaler()
X = scaler.fit_transform(X)
print("Features Normalization Completed : "+str(X))

Features Normalization Completed : [[-1.41421356  1.41421356  1.41421356 -0.47937669  1.02899151 -1.41421356
   0.70710678 -1.41421356 -1.41421356 -0.15899968]
 [-0.70710678 -0.70710678  0.          1.99957233  0.17149859 -0.70710678
  -1.41421356  0.70710678 -0.70710678 -1.21899756]
 [ 0.          0.         -1.41421356 -0.52406607  1.02899151  0.
  -0.70710678 -0.70710678  0.          1.69599661]
 [ 0.70710678 -1.41421356 -0.70710678 -0.47433547 -0.68599434  0.70710678
   0.          1.41421356  0.70710678 -0.68899862]
 [ 1.41421356  0.70710678  0.70710678 -0.52179411 -1.54348727  1.41421356
   1.41421356  0.          1.41421356  0.37099926]]


In [24]:
#split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3)
print("80% Training Size = "+str(X_train.shape[0]))
print("20% Testing Size = "+str(X_test.shape[0]))

80% Training Size = 3
20% Testing Size = 2


In [25]:
#function to calculate all metrics
def calculateMetrics(algorithm, y_test, predict):
    global graph
    a = accuracy_score(y_test,predict)*100
    p = precision_score(y_test, predict,average='macro') * 100
    r = recall_score(y_test, predict,average='macro') * 100
    f = f1_score(y_test, predict,average='macro') * 100
    a = round(a, 3)
    p = round(p, 3)
    r = round(r, 3)
    f = round(f, 3)
    print(algorithm+" Accuracy  : "+str(a))
    print(algorithm+" Precision : "+str(p))
    print(algorithm+" Recall    : "+str(r))
    print(algorithm+" FSCORE    : "+str(f))

In [26]:
#training Randon Forest 
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
#performing prediction on test data
predict = rf.predict(X_test)
#call this function to calculate accuracy and other metrics
calculateMetrics("Random Forest", y_test, predict)

Random Forest Accuracy  : 50.0
Random Forest Precision : 25.0
Random Forest Recall    : 50.0
Random Forest FSCORE    : 33.333


c:\users\admin\appdata\local\programs\python\python37\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
#kafka producer to send stream
def delivery_report(err, msg):
    if err is not None:
        print(f'Message delivery failed: {err}')
    else:
        print(f'Message delivered to {msg.topic()} [{msg.partition()}]')
#creating producer
p = Producer({'bootstrap.servers': 'localhost:9092'})
#loading dataset
dataset = pd.read_csv("Dataset/BankingTransactionDataSet.csv")
dataset = dataset.values
#producer publishing dataset topics to bank transaction
for i in range(len(dataset)):
    data = ""
    for j in range(len(dataset[i])):
        data += str(dataset[i,j])+"#"
    if len(data) > 0:
        data = data[0:len(data)-1]
    p.produce('BankTransaction', data, callback=delivery_report)
    p.flush()
p.produce('BankTransaction', "exit", callback=delivery_report)
p.flush()

In [8]:
def consume():
    cols = ['transaction_id','sender_account_id','recipient_account_id','amount','payment_mode','timestamp','sender_location',
            'recipient_location','device_fingerprint','transaction_frequency']
    conf = {'bootstrap.servers': 'localhost:9092', 'group.id': 'sensor_stream_processor', 'auto.offset.reset': 'earliest'}
    consumer = Consumer(conf)
    # Subscribe to the BankTransaction
    consumer.subscribe(['BankTransaction'])
    # consume to stream data and process to cassandra
    for i in range(0, 10):
        msg = consumer.poll(1.0)
        if msg is None:
            # No message received in the last poll interval
            continue
        elif msg.error():
            # Handle any errors that occurred while polling for messages
            raise KafkaException(msg.error())
        else:
            msgs =  msg.value().decode('utf-8')
            print(msgs)
            if msgs == "exit":
                break
            else:
                data = msgs.split("#")
                values = []
                values.append([data[0], data[1], data[2], float(data[3]), data[4], data[5], data[6], data[7], data[8], int(data[9])])
                values = pd.DataFrame(values, columns=cols)
                temp = pd.DataFrame(values, columns=cols)
                for i in range(len(label_encoder)):
                    le = label_encoder[i]
                    values[le[0]] = pd.Series(le[1].transform(values[le[0]].astype(str)))#encode all str columns to numeric
                values = values.values
                values = scaler.transform(values)
                predict = rf.predict(values)[0]
                temp['Predicted'] = predict
                temp

In [9]:
from threading import Thread
thread = Thread(target = consume)
thread.start()
thread.join()